In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(ggrepel)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(splines)
    library(rmcorr)
    library(ComplexHeatmap)
    library(circlize)
    library(viridis)
    library(tidyr)
    library(purrr)
})

# Load data

In [ ]:
zfish_dir = '../data/cds_ref_v2.2.0_glmm/'

cds_fish = load_monocle_objects(zfish_dir)

In [ ]:
metadata = as.data.frame(colData(cds_fish))
umap_coordinates = reducedDims(cds_fish)[['UMAP']]

metadata = metadata %>%
             mutate(UMAP1 = umap_coordinates[,1],
                    UMAP2 = umap_coordinates[,2], 
                    UMAP3 = umap_coordinates[,3]) 

In [ ]:
## Timepoint = 60 was omitted due to low UMI counts
metadata_filtered = metadata %>%
            filter(normalized_nuc_score_remapped > 0) %>%
            filter(!grepl("doublet|unknown", cell_type)) %>%
            filter(!is.na(timepoint) & timepoint != 60 & !is.na(cell_type))

In [ ]:
quantile(metadata_filtered$score_pred_glmm_0.7_0.4_remapped, c(0.05, 0.1, 0.5, 0.9, 0.95))

In [ ]:
## Supplementary Figure 28A

options(repr.plot.width=8, repr.plot.height=7, repr.plot.res=300)
metadata_filtered %>%
    ggplot(., aes(x = UMAP1, y = UMAP3)) + 
        geom_point(aes(color = score_pred_glmm_0.7_0.4_remapped), alpha = 0.8,
                   size = I(0.2), stroke= I(0.2 / 2)) +
        scale_colour_gradient2(low = muted("blue"), mid = "grey90", high = muted("red"), midpoint = 0.26,
                               limits = c(0.12, 0.40), oob=scales::squish) + 
        labs(color = "Score") +  
        monocle3:::monocle_theme_opts() + 
        theme(aspect.ratio = 1)

In [ ]:
label_data <- metadata_filtered %>%
                group_by(tissue) %>%
                summarize(UMAP1 = median(UMAP1, na.rm = TRUE),
                          UMAP3 = median(UMAP3, na.rm = TRUE)) 


## Supplementary Figure 28B

options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
metadata_filtered %>%
    ggplot(., aes(x = UMAP1, y = UMAP3)) + 
        geom_point(aes(color = tissue), size = I(0.2), stroke= I(0.2 / 2)) +
        geom_text_repel(data = label_data, 
                aes(label = str_wrap(str_replace_all(tissue, "_", " "), width = 10)),
                    size = 4,    
                    color = "black",
                    bg.color = "white", 
                    bg.r = 0.15,
                    segment.color = "black", 
                    segment.size = 0.2,
                    box.padding = 0.7,
                    max.overlaps = Inf  
                ) +
        scale_color_viridis_d(option = "turbo") + 
        monocle3:::monocle_theme_opts() +

        theme(
            axis.title = element_text(color="black", size = 9), 
            axis.text.x = element_text(color="black", size = 9), 
            axis.text.y = element_text(color = "black", size = 9),
            panel.grid.major.x = element_blank(), 
            axis.ticks = element_line(color = "black"),
            legend.position = "none")

In [ ]:
data_summary <- function(x) {
   m <- mean(x)
   ymin <- m-sd(x)
   ymax <- m+sd(x)
   return(c(y=m,ymin=ymin,ymax=ymax))
}

## Supplementary Figure 28C
options(repr.plot.width=3.75, repr.plot.height=4, repr.plot.res=300)
metadata_filtered %>%
    group_by(timepoint, embryo_ID) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4_remapped), n = n()) %>%
    filter(n >= 100) %>%
    ggplot(., aes(y = med_score, x = as.factor(timepoint))) +  
        geom_boxplot(aes(fill = timepoint), fatten = 0.75, lwd = 0.75, outlier.shape = NA) +
        scale_fill_viridis_c(begin = 0.2) + 
        coord_cartesian(ylim = c(NA, 0.275)) + 
        labs(x = "", y = "") + 
        monocle3:::monocle_theme_opts() + theme(legend.position = "none") +
        theme(axis.text.x = element_text(color = "black", size = 10),
              legend.position = "none",
              axis.line = element_line(color = "black"), 
              axis.text.y = element_text(color = "black", size = 10),
              axis.ticks = element_line(color = "black"),
              axis.ticks.x = element_line(color = "black"))

## Heatmap by tissue

In [ ]:
projection_group_timepoint = metadata_filtered %>%
    group_by(timepoint, tissue, embryo_ID) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4_remapped)) %>%
    group_by(timepoint, tissue) %>%
    summarise(med_score = median(med_score)) %>%
    as.data.frame() %>%
    tidyr::spread(timepoint, med_score) %>%
    column_to_rownames(var = "tissue")

In [ ]:
rownames(projection_group_timepoint) = stringr::str_to_title(rownames(projection_group_timepoint))
rownames(projection_group_timepoint) <- stringr::str_replace_all(
        rownames(projection_group_timepoint), 
        c("Cns Other" = "CNS Other", 
        "Lpm Derivatives" = "LPM Derivatives")
)
projection_group_timepoint

In [ ]:
quantile(na.omit(unlist(c(projection_group_timepoint))), 
         c(0.02, 0.05, 0.1, 0.5, 0.9, 0.95, 0.98))

In [ ]:
## Figure 7D

options(repr.plot.width=7.25, repr.plot.height=6.5, repr.plot.res=300)
ht = Heatmap(projection_group_timepoint,
        cluster_columns = FALSE, column_names_rot = 0, column_names_centered = TRUE,
        show_column_names = T,
        na_col = "grey40", 
        name = " ", row_names_gp = gpar(fontsize = 11), column_names_gp = gpar(fontsize = 10),
        col = colorRamp2(c(0.16, 0.23, 0.30), c(muted("blue"), "white", muted("red"))),
        rect_gp = gpar(col = "grey70", lwd = 1),
        heatmap_legend_param = list( title_position = "lefttop",
               direction = "vertical"))

draw(ht, heatmap_legend_side = "right", annotation_legend_side = "top")

In [ ]:
## Figure 7E
options(repr.plot.width=5, repr.plot.height=4, repr.plot.res=300)

traj = "skeletal muscle"

df_sub = metadata_filtered %>%
    filter(projection_group == traj)

min_lim = quantile(df_sub$score_pred_glmm_0.7_0.4_remapped, 0.3)
mid_lim = quantile(df_sub$score_pred_glmm_0.7_0.4_remapped, 0.7)
max_lim = quantile(df_sub$score_pred_glmm_0.7_0.4_remapped, 0.9)

df_sub %>%
    ggplot(., aes(x = UMAP3, y = UMAP2)) +
        geom_point(aes(color = score_pred_glmm_0.7_0.4_remapped), 
                   size = I(0.3), stroke= I(0.3 / 2), alpha = 0.6) +
        ggtitle(traj) + 
        scale_colour_gradient2(name = "Score",
                               low = muted("blue"), mid = "grey90", high = muted("red"), 
                               midpoint = mid_lim,
                               limits = c(min_lim, max_lim),
                               oob=scales::squish) + 
        monocle3:::monocle_theme_opts() + 
    theme(aspect.ratio = 1)


In [ ]:
## Figure 7F

df_sub = metadata_filtered %>%
    filter(tissue == "fast muscle") %>%
    group_by(timepoint, embryo_ID) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4_remapped), n = n()) %>%
    filter(n >= 10)


options(repr.plot.width=4, repr.plot.height=2.8, repr.plot.res=300)
df_sub %>%
    arrange(-med_score) %>%
    ggplot(., aes(x = as.factor(timepoint), y = med_score)) + 
        geom_boxplot(aes(fill = timepoint), outlier.shape = NA, lwd = 0.75, fatten = 0.75) + 
        scale_fill_viridis_c(begin = 0.2) + 
        scale_y_continuous(labels = label_number(accuracy = 0.01), limits = c(NA, 0.31)) + 
        labs(x = "Timepoint (hpf)", y = "Nucleolar fraction score") + 
        monocle3:::monocle_theme_opts() +
        theme(
            axis.text.x = element_text(color = "black", size = 10),
            axis.text.y = element_text(color = "black", size = 11),
            axis.title = element_text(size = 12),
            legend.text = element_text(size = 9, lineheight = 0.9),
            legend.position = "none",
            legend.key.height = unit(0.6, "cm")
        )

df_sub = metadata_filtered %>%
    filter(tissue == "slow muscle") %>%
    group_by(timepoint, embryo_ID) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4_remapped), n = n()) %>%
    filter(n >= 10) %>%
    group_by(timepoint) %>% add_tally() %>% filter(nn > 2)

options(repr.plot.width=4, repr.plot.height=2.8, repr.plot.res=300)
df_sub %>%
    arrange(-med_score) %>%
    ggplot(., aes(x = as.factor(timepoint), y = med_score)) + 
        geom_boxplot(aes(fill = timepoint), outlier.shape = NA, lwd = 0.75, fatten = 0.75) + 
        scale_fill_viridis_c(begin = 0.2) +  
        ylim(c(0.12, 0.19)) + 
        labs(x = "Timepoint (hpf)", y = "Nucleolar fraction score") + 
        monocle3:::monocle_theme_opts() +
        theme(
            axis.text.x = element_text(color = "black", size = 10),
            axis.text.y = element_text(color = "black", size = 11),
            axis.title = element_text(size = 14),
            legend.text = element_text(size = 9, lineheight = 0.9),
            legend.position = "none",
            legend.key.height = unit(0.6, "cm")
        )

## Supplementary Figure 28D

In [ ]:
n_filter = 10

cell_type_df <- metadata_filtered %>%
    group_by(tissue, cell_type, timepoint, embryo_ID) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4_remapped),
              n = n(), .groups = "drop") %>%
    filter(n >= n_filter) %>%
    arrange(-med_score)

In [ ]:
cell_type_df_early = cell_type_df %>% filter(timepoint <= 26)

celltype_order_early = cell_type_df_early %>%
                    group_by(cell_type) %>%
                    summarise(mm_score = median(med_score)) %>%
                    arrange(-mm_score) %>%
                    pull(cell_type)

cell_type_df_early = cell_type_df_early %>% 
                        mutate(cell_type = factor(cell_type, levels = celltype_order_early))


cell_type_df_late = cell_type_df %>% filter(timepoint >= 72)

celltype_order_late = cell_type_df_late %>%
                    group_by(cell_type) %>%
                    summarise(mm_score = median(med_score)) %>%
                    arrange(-mm_score) %>%
                    pull(cell_type)

cell_type_df_late = cell_type_df_late %>% 
                        mutate(cell_type = factor(cell_type, levels = celltype_order_late))

cell_type_df_mid = cell_type_df %>% filter(timepoint >= 28 & timepoint <= 48)

celltype_order_mid = cell_type_df_mid %>%
                    group_by(cell_type) %>%
                    summarise(mm_score = median(med_score)) %>%
                    arrange(-mm_score) %>%
                    pull(cell_type)

cell_type_df_mid = cell_type_df_mid %>% 
                        mutate(cell_type = factor(cell_type, levels = celltype_order_mid))

In [ ]:
cell_type_df_early
cell_type_df_mid
cell_type_df_late

In [ ]:
options(repr.plot.width=5, repr.plot.height=4, repr.plot.res=400)
cell_type_df_early %>%
    filter(cell_type %in% celltype_order_early[1:10]) %>%
    ggplot(., aes(x = cell_type, y = med_score)) + 
        geom_boxplot(lwd = 0.75, fatten = 0.75) + 
        scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 34)) +
        monocle3:::monocle_theme_opts() +
        #ylim(c(NA, 0.75)) + 
        labs(x = "", y = "") + 
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 9, color = "black"), 
              axis.text.y = element_text(size = 13, color = "black"), 
              legend.position = "none",
              plot.margin = margin(l = 60))


options(repr.plot.width=5, repr.plot.height=4.25, repr.plot.res=400)
cell_type_df_early %>%
    filter(cell_type %in% tail(celltype_order_early, 10)) %>%
    ggplot(., aes(x = cell_type, y = med_score)) + 
        geom_boxplot(lwd = 0.75, fatten = 0.75) +
        scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 35)) + 
        ylim(c(NA, 0.23)) + 
        monocle3:::monocle_theme_opts() +
        #ylim(c(NA, 0.75)) + 
        labs(x = "", y = "") + 
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 9, color = "black"), 
              axis.text.y = element_text(size = 13, color = "black"), 
              legend.position = "none",
              plot.margin = margin(l = 60))

In [ ]:
options(repr.plot.width=5, repr.plot.height=4, repr.plot.res=400)
cell_type_df_late %>%
    filter(cell_type %in% celltype_order_late[1:10]) %>%
    ggplot(., aes(x = cell_type, y = med_score)) + 
        geom_boxplot(lwd = 0.75, fatten = 0.75) +
        monocle3:::monocle_theme_opts() +
        #ylim(c(NA, 0.75)) + 
        scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 35)) + 
        labs(x = "", y = "") + 
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 9, color = "black"), 
              axis.text.y = element_text(size = 13, color = "black"), 
              legend.position = "none",
              plot.margin = margin(l = 60))



options(repr.plot.width=5.5, repr.plot.height=4.5, repr.plot.res=400)
cell_type_df_late %>%
    filter(cell_type %in% tail(celltype_order_late, 10)) %>%
    ggplot(., aes(x = cell_type, y = med_score)) + 
        geom_boxplot(lwd = 0.75, fatten = 0.75) + 
        scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 35)) + 
        monocle3:::monocle_theme_opts() +
        ylim(c(NA, 0.2)) + 
        labs(x = "", y = "") + 
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 9, color = "black"), 
              axis.text.y = element_text(size = 13, color = "black"), 
              legend.position = "none",
              plot.margin = margin(l = 60))


In [ ]:
options(repr.plot.width=5, repr.plot.height=4.5, repr.plot.res=400)
cell_type_df_mid %>%
    filter(cell_type %in% celltype_order_mid[1:10]) %>%
    ggplot(., aes(x = cell_type, y = med_score)) + 
        geom_boxplot(lwd = 0.75, fatten = 0.75) +
        ylim(c(NA, 0.42)) + 
        monocle3:::monocle_theme_opts() +
        scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 35)) + 
        labs(x = "", y = "") + 
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 9, color = "black"), 
              axis.text.y = element_text(size = 13, color = "black"), 
              legend.position = "none",
              plot.margin = margin(l = 60))

options(repr.plot.width=5, repr.plot.height=4.5, repr.plot.res=400)
cell_type_df_mid %>%
    filter(cell_type %in% tail(celltype_order_mid, 10)) %>%
    ggplot(., aes(x = cell_type, y = med_score)) + 
        geom_boxplot(lwd = 0.75, fatten = 0.75) + 
        ylim(c(NA, 0.2)) + 
        monocle3:::monocle_theme_opts() +
        scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 35)) + 
        labs(x = "", y = "") + 
        theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1, size = 9, color = "black"), 
              axis.text.y = element_text(size = 13, color = "black"), 
              legend.position = "none",
              plot.margin = margin(l = 60))

## Figure 7G

In [ ]:
n_filter = 20
safe_cor_test <- possibly(cor.test, otherwise = NULL)

calculate_tissue_cor <- function(data, target_col, score_name) {
    target_sym <- sym(target_col)
    
    data %>%
        filter(!!target_sym > 0) %>%
        group_by(tissue, cell_type, embryo_ID) %>%
        summarise(
            med_score = median(score_pred_glmm_0.7_0.4_remapped),
            med_target = median(!!target_sym),
            n = n(),
            .groups = "drop"
        ) %>%
        filter(n >= n_filter) %>%
        group_by(tissue, cell_type) %>%
        summarise(
            med_score = median(med_score),
            med_target = median(med_target),
            .groups = "drop"
        ) %>%
        group_by(tissue) %>%
        filter(n() >= 5) %>%
        nest() %>%
        mutate(
            cor_model = map(data, ~ suppressWarnings(
                safe_cor_test(.x$med_score, .x$med_target, method = "spearman")
            ))
        ) %>%
        filter(!map_lgl(cor_model, is.null)) %>%
        mutate(
            score_type = score_name,
            spearman_rho = map_dbl(cor_model, "estimate"),
            p_value = map_dbl(cor_model, "p.value"),
            n_points = map_int(data, nrow)
        ) %>%
        select(score_type, tissue, spearman_rho, p_value, n_points)
}

upr_tissue_cor     <- calculate_tissue_cor(metadata_filtered, "upr_score", "UPR")
granule_tissue_cor <- calculate_tissue_cor(metadata_filtered, "granule_score", "Stress granule")

combined_tissue_cor <- bind_rows(upr_tissue_cor,granule_tissue_cor) %>%
    arrange(score_type, desc(spearman_rho))

combined_tissue_cor

In [ ]:
combined_tissue_cor %>%
    filter(spearman_rho < 0)

In [ ]:
saveRDS(combined_tissue_cor, "../data/combined_tissue_cor_zebrafish.RDS")

In [ ]:
df_upr <- metadata_filtered %>%
    filter(upr_score > 0) %>%
    group_by(tissue, cell_type, embryo_ID) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4_remapped),
              med_target = median(upr_score),
              n = n(), .groups = "drop") %>%
    filter(n >= n_filter) %>%
    group_by(tissue, cell_type) %>%
    summarise(med_score = median(med_score),
              med_target = median(med_target),
              n = n(), .groups = "drop") %>%
    mutate(score_type = "UPR")

df_granule <- metadata_filtered %>%
    filter(granule_score > 0) %>%
    group_by(tissue, cell_type, embryo_ID) %>%
    summarise(med_score = median(score_pred_glmm_0.7_0.4_remapped),
              med_target = median(granule_score),
              n = n(), .groups = "drop") %>%
    filter(n >= n_filter) %>%
    group_by(tissue, cell_type) %>%
    summarise(med_score = median(med_score),
              med_target = median(med_target),
              n = n(), .groups = "drop") %>%
    mutate(score_type = "Stress granule")

df_combined <- bind_rows(df_upr, df_granule)

In [ ]:
options(repr.plot.width=3, repr.plot.height=4, repr.plot.res=300)

ggplot(df_combined, aes(x = med_score, y = med_target, color = score_type)) + 
    geom_point(size = 0.8, alpha = 0.5) + 
    scale_color_manual(values = c("UPR" = "#c51b7d", "Stress granule" = "#4d9221")) +

    geom_smooth(method = "lm", se = FALSE, linewidth = 1.2) +
    
    labs(x = "", y = "", color = "") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "top",
          axis.text.x = element_text(color = "black", size = 14), 
          axis.text.y = element_text(color = "black", size = 14), 
          axis.title = element_text(color = "black", size = 14), 
          axis.ticks = element_line(color = "black"))

In [ ]:
correlation_stats <- df_combined %>%
  group_by(score_type) %>%
  summarise(
    pearson_r = cor(med_score, med_target, method = "pearson", use = "complete.obs"),
    p_value = cor.test(med_score, med_target, method = "pearson")$p.value,
    n_cell_type = n()
  )

print(correlation_stats)

In [ ]:
length(unique(df_combined$cell_type))

## Figure 7H

In [ ]:
cor_mouse = readRDS("../data/combined_tissue_cor_mouse.RDS") %>%
                mutate(org = "Mouse")
cor_human = readRDS("../data/combined_tissue_cor_human_fetal_atlas.RDS") %>%
                mutate(org = "Human")
cor_zebrafish = readRDS("../data/combined_tissue_cor_zebrafish.RDS") %>%
                mutate(org = "Zebrafish")

In [ ]:
combined_all <- bind_rows(cor_mouse, cor_human, cor_zebrafish) %>%
    mutate(org = factor(org, levels = c("Mouse", "Zebrafish", "Human")),
           score_type = factor(score_type, levels = c("UPR", "Stress granule")))


options(repr.plot.width=3.5, repr.plot.height=4, repr.plot.res=300)
ggplot(combined_all, aes(x = score_type, y = spearman_rho, fill = org)) + 
    geom_boxplot(alpha = 0.7, color = "black", outlier.shape = NA, width = 0.6,
                 lwd = 0.75, fatten = 0.75) + 
    
    geom_point(aes(color = org), position = position_jitterdodge(jitter.width = 0.2), 
               size = 1, alpha = 0.7, show.legend = FALSE) + 
    
    scale_fill_manual(values = c("Mouse" = "#1b9e77", "Human" = "#d95f02", "Zebrafish" = "#7570b3")) +
    scale_color_manual(values = c("Mouse" = "black", "Human" = "black", "Zebrafish" = "black")) +

    labs(x = "", y = "Spearman correlation", fill = "") + 
    monocle3:::monocle_theme_opts() + 
    theme(legend.position = "bottom", 
          axis.text.x = element_text(color = "black", size = 13), 
          axis.text.y = element_text(color = "black", size = 13), 
          axis.title.y = element_text(color = "black", size = 14),
          axis.ticks = element_line(color = "black"))